<a href="https://colab.research.google.com/github/pinakm9/fp-solvers/blob/master/colab-solvers/regular_stationary_L63.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# run this cell to download data and necessary modules
import os, shutil
repo = 'fp-solvers'
if os.path.isdir(repo):
  shutil.rmtree(repo)
!git clone https://github.com/pinakm9/non-grad3D.git
# add modules folder to Python's search path
import sys
sys.path.insert(0, repo + '/modules')
# import the necessary modules
import numpy as np
import tensorflow as tf
import lss_solver as lss
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
DTYPE = 'float32'

Cloning into 'non-grad3D'...
remote: Enumerating objects: 13457, done.
remote: Counting objects: 100% (3368/3368), done.
remote: Compressing objects: 100% (1733/1733), done.
remote: Total 13457 (delta 1643), reused 3349 (delta 1627), pack-reused 10089
Receiving objects: 100% (13457/13457), 675.55 MiB | 46.04 MiB/s, done.
Resolving deltas: 100% (6645/6645), done.
Checking out files: 100% (13330/13330), done.


**Define the equation through the $\mathcal L_{\log}$ operator**

In [ ]:
dim=3
alpha, beta, rho = 10., 8./3., 28.
sigma = 10.
D = sigma**2 / 2.

low=[-30., -40., 0.]
high=[30., 40., 70.]
domain = [low, high]
save_folder = '{}/non-grad3D/data/L63'.format(repo)

def mu(x, y, z):
  p = alpha * (y - x) 
  q = x * (rho - z) - y 
  r = x * y - beta * z
  return p, q, r


@tf.function
def diff_log_op(f, x, y, z):
    with tf.GradientTape(persistent=True) as tape:
        tape.watch([x, y, z])
        f_ = f(x, y, z)
        f_x, f_y, f_z = tape.gradient(f_, [x, y, z])
    f_xx = tape.gradient(f_x, x)
    f_yy = tape.gradient(f_y, y)
    f_zz = tape.gradient(f_z, z)
    p, q, r = mu(x, y, z)
    return -(p*f_x + q*f_y + r*f_z) + (alpha + beta + 1.) + D*(f_xx + f_yy + f_zz + f_x**2 + f_y**2 + f_z**2) 

**Set up experiment parameters and learn the stationary distribution**

In [ ]:
learning_rate = tf.keras.optimizers.schedules.PiecewiseConstantDecay([1000, 2000, 10000, 50000], [5e-3, 1e-3, 5e-4, 1e-4, 1e-5])
optimizer = tf.keras.optimizers.Adam(learning_rate)
model_path = '{}/data/L63/800k/L63'.format(repo)
solver = lss.LogSteadyStateSolver(num_nodes=50, num_blocks=3, dtype=DTYPE, name='L63'.format(dim), diff_log_op=diff_log_op, optimizer=optimizer, domain=domain, model_path=model_path)
solver.learn(epochs = 200000, n_sample = 1000, save_folder=save_folder)

Streaming output truncated to the last 5000 lines.
150000    0.023266         3415.3692
150010    0.013934         3415.6100
150020    0.016437         3415.8315
150030    0.018295         3416.0539
150040    0.014545         3416.2776
150050    0.015379         3416.5016
150060    0.015774         3416.7253
150070    0.014074         3416.9682
150080    0.015734         3417.1919
150090    0.016788         3417.4248
150100    0.015062         3417.6483
150110    0.017592         3417.8686
150120    0.013535         3418.0920
150130    0.017358         3418.3182
150140    0.013088         3418.5421
150150    0.014914         3418.7664
150160    0.016054         3419.0006
150170    0.011666         3419.2292
150180    0.012761         3419.4520
150190    0.015658         3419.6773
150200    0.018079         3419.9003
150210    0.018305         3420.1233
150220    0.018576         3420.3600
150230    0.016967         3420.5862
150240    0.017556         3420.8083
150250    0.016037      

**Visualize $p(x, y)$**

In [ ]:
N = 50 # <--- resolution of heatmap
x = np.linspace(x_min, x_max, num=N)
y = np.linspace(y_min, y_max, num=N)

M = 50 # <--- (even) number of points to be used in 1d integration with Simpson's 1/3 rule
prob = np.zeros((N, N))
one = tf.ones((M, 1))
z = np.linspace(z_min, z_max, num=M, endpoint=True).reshape(-1, 1)
w = np.array([2. if i%2==0 else 4. for i in range(M)])
w[0], w[-1] = 1., 1.
h = (z_max - z_min) / (3. * M)
for i in range(N):
  for j in range(N):
    prob[i, j] = h * (tf.exp(n_theta(x[i]*one, y[j]*one, z)).numpy().reshape((-1,)) * w).sum()
    #pass

x, y = np.meshgrid(x, y)
fig = plt.figure(figsize=(16, 8))
# learned p(x, y)
ax_l = fig.add_subplot(121)
im = ax_l.pcolormesh(x, y, prob, cmap='inferno')
fig.colorbar(im)
ax_l.set_xlabel('x')
ax_l.set_ylabel('y')
ax_l.set_title(r'Learned $p(x, y)$')
# Monte-Carlo p(x, y)
ax_mc = fig.add_subplot(122)
ax_mc.imshow(plt.imread('fp-solvers/Stationary/L63/sigma_10/plots/p_xy_mc.png'), aspect='auto')
ax_mc.set_axis_off()
plt.show()

NameError: ignored

**Visualize $p(y, z)$**

In [ ]:
N = 50 # <--- resolution of heatmap
z = np.linspace(z_min, z_max, num=N)
y = np.linspace(y_min, y_max, num=N)

M = 50 # <--- (even) number of points to be used in 1d integration with Simpson's 1/3 rule
prob = np.zeros((N, N))
one = tf.ones((M, 1))
x = np.linspace(x_min, x_max, num=M, endpoint=True).reshape(-1, 1)
w = np.array([2. if i%2==0 else 4. for i in range(M)])
w[0], w[-1] = 1., 1.
h = (x_max - x_min) / (3. * M)
for i in range(N):
  for j in range(N):
    prob[i, j] = h * (tf.exp(n_theta(x, y[i]*one, z[j]*one)).numpy().reshape((-1,)) * w).sum()
    #pass

y, z = np.meshgrid(y, z)
fig = plt.figure(figsize=(16, 8))
# learned p(y, z)
ax_l = fig.add_subplot(121)
im = ax_l.pcolormesh(y, z, prob, cmap='inferno')
fig.colorbar(im)
ax_l.set_xlabel('y')
ax_l.set_ylabel('z')
ax_l.set_title(r'Learned $p(y, z)$')
# Monte-Carlo p(y, z)
ax_mc = fig.add_subplot(122)
ax_mc.imshow(plt.imread('fp-solvers/Stationary/L63/sigma_10/plots/p_yz_mc.png'), aspect='auto')
ax_mc.set_axis_off()
plt.show()

**Visualize $p(x,z)$**

In [ ]:
N = 50 # <--- resolution of heatmap
x = np.linspace(x_min, x_max, num=N)
z = np.linspace(z_min, z_max, num=N)

M = 50 # <--- (even) number of points to be used in 1d integration with Simpson's 1/3 rule
prob = np.zeros((N, N))
one = tf.ones((M, 1))
y = np.linspace(y_min, y_max, num=M, endpoint=True).reshape(-1, 1)
w = np.array([2. if i%2==0 else 4. for i in range(M)])
w[0], w[-1] = 1., 1.
h = (z_max - z_min) / (3. * M)
for i in range(N):
  for j in range(N):
    prob[i, j] = h * (tf.exp(n_theta(x[i]*one, y, z[j]*one)).numpy().reshape((-1,)) * w).sum()
    #pass

x, z = np.meshgrid(x, z)
fig = plt.figure(figsize=(16, 8))
# learned p(x, z)
ax_l = fig.add_subplot(121)
im = ax_l.pcolormesh(x, z, prob, cmap='inferno')
fig.colorbar(im)
ax_l.set_xlabel('x')
ax_l.set_ylabel('z')
ax_l.set_title(r'Learned $p(x, z)$')

# Monte-Carlo p(x, z)
ax_mc = fig.add_subplot(122)
ax_mc.imshow(plt.imread('fp-solvers/Stationary/L63/sigma_10/plots/p_xz_mc.png'), aspect='auto')
ax_mc.set_axis_off()
plt.show()

**Investigate the size of $θ$**

In [ ]:
n_theta.summary()

**Visualize the reference points**

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(x_ref, y_ref, z_ref)
plt.show()